# Classifier experiments v2: minority classes, seeds, and our reranker's evidence

Follows `results/classifier_v1.md`, where the trained classifier beat the paper's accuracy
(0.649 vs 0.584) but not its macro-F1 (0.487 vs 0.524) because it under-predicts the two
minority classes. This notebook runs two groups of 3 seeds each, all with `--class-weight inverse`:

- **A. Paper's evidence.** Trained on BM25+MonoT5 top-20, the same as v1 but with stronger class weights.
- **B. Our reranker's evidence.** Trained on top-20 from our PubMedBERT reranker. The training
  split is ranked by 3-fold cross-validated rerankers, so every training citation's evidence
  comes from a reranker that never saw it, matching what happens at test time.

Both groups are scored on test with both kinds of evidence. A against B is the experiment that
asks whether better retrieval helps once the classifier is trained on it.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all, and
allow Google Drive access when asked. Total time is about 2 hours on a T4. Every finished step is
saved to `MyDrive/CiteAudit_runs`, so if Colab disconnects, Run all again and it resumes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs'
import os; os.makedirs(f'{RES}/cv', exist_ok=True)

## Setup

Clones this repository and the authors' data (needed only to build group B's training evidence),
and joins the raw annotations to the claims so the 3-way labels are correct.

In [ ]:
%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19" sentencepiece protobuf
!bash bootstrap.sh > /dev/null
!unzip -q -o upstream/Citation-Integrity/Data/annotations.zip -d upstream/annotations_extracted -x "__MACOSX/*"
!python scripts/align_annotations.py --split train --out results/annotations/train.jsonl | tail -1

import subprocess, json, statistics
def run(cmd):
    """Run a shell command, show its output without the model-loading noise, stop on failure."""
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS', 'Loading weights')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

def train_classifier(group, train_dir, dev_dir, seed):
    out = f'{RES}/{group}_s{seed}'
    if os.path.exists(f'{out}/ours_20_test.jsonl'):
        print(f'{group} seed {seed}: already done'); return
    run(f'python scripts/train_classifier.py --train {train_dir} --dev {dev_dir} '
        f'--out checkpoints/{group}_s{seed} --pred-dir {out} --class-weight inverse --seed {seed} '
        f'--predict test=data_train/target/paper_20_test --predict test=data_train/target/ours_20_test '
        f'--amp --eval-batch 16')
    run(f'cp checkpoints/{group}_s{seed}/config.json {out}/ && rm -rf checkpoints/{group}_s{seed}')

## A. Paper's evidence, 3 seeds (about 35 minutes)

In [ ]:
for seed in (0, 1, 2):
    train_classifier('A_paper', 'data_train/target/paper_20_train', 'data_train/target/paper_20', seed)

## B1. Our reranker's evidence for the training split, by 3-fold cross-validation (about 1 hour)

Folds are cut by reference article, so a held-out fold shares no article text with its
reranker's training data. Same recipe as the reranker in the README.

In [ ]:
for k in range(3):
    f = f'{RES}/cv/train_fold{k}.jsonl'
    if os.path.exists(f):
        print(f'fold {k}: already done'); continue
    run(f'python scripts/train_reranker.py --epochs 1 --negatives 4 --groups-per-batch 4 --seed 0 '
        f'--folds 3 --fold {k} --train-split train --out checkpoints/cv_f{k} '
        f'--predict train --rankings-out {f}.part')
    os.rename(f'{f}.part', f)
    run(f'rm -rf checkpoints/cv_f{k}')
run(f'cat {RES}/cv/train_fold0.jsonl {RES}/cv/train_fold1.jsonl {RES}/cv/train_fold2.jsonl > results/train_cv_ce.jsonl')
run('python scripts/retrieval_harness.py --split train --ranker file --rankings-in results/train_cv_ce.jsonl '
    f'--metrics-out {RES}/cv/train_cv_ce.metrics.json | tail -3')
run('python scripts/export_for_multivers.py --rankings results/train_cv_ce.jsonl --split train --topk 20 '
    '--out data_train/target/ours_20_train_cv')

## B2. Our reranker's evidence, 3 seeds (about 35 minutes)

Dev is our reranker's dev evidence (`ours_20`), so the epoch is chosen on the same kind of input.

In [ ]:
for seed in (0, 1, 2):
    train_classifier('B_ours', 'data_train/target/ours_20_train_cv', 'data_train/target/ours_20', seed)

## Results

Mean and standard deviation over the 3 seeds. The paper's top-20 test figures are accuracy 0.59
and macro-F1 0.52; the released MultiVerS checkpoint gives 0.584 / 0.524 on the paper's evidence.

In [ ]:
LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(path):
    rows = [json.loads(l) for l in open(path)]
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1

cols = ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']
print(f'{"trained on":10s} {"test evidence":14s} ' + ' '.join(f'{c:>14s}' for c in cols))
summary = {}
for group in ('A_paper', 'B_ours'):
    for test in ('paper_20_test', 'ours_20_test'):
        runs = [scores(f'{RES}/{group}_s{s}/{test}.jsonl') for s in (0, 1, 2)
                if os.path.exists(f'{RES}/{group}_s{s}/{test}.jsonl')]
        if not runs: continue
        mean = [statistics.mean(x) for x in zip(*runs)]
        sd = [statistics.stdev(x) if len(runs) > 1 else 0. for x in zip(*runs)]
        summary[f'{group}/{test}'] = {'n_seeds': len(runs), 'mean': mean, 'sd': sd}
        print(f'{group:10s} {test:14s} ' + ' '.join(f'{m:.3f} ± {s:.3f}' for m, s in zip(mean, sd)))
json.dump(summary, open(f'{RES}/summary.json', 'w'), indent=2)

## Download

Zips everything in `MyDrive/CiteAudit_runs` (predictions, training logs, CV rankings and their
metrics; no model weights). Put the unzipped folder in the repository root and tell Claude.

In [ ]:
!cd /content/drive/MyDrive && zip -qr /content/CiteAudit_runs.zip CiteAudit_runs
from google.colab import files
files.download('/content/CiteAudit_runs.zip')